# EDA — Green Turning Point (GTP)
## Análisis Exploratorio de Datos · Justificación del Pipeline ETL

**Universidad Europea · Big Data I · 3º Grado GIAMAD**  
**Autores:** Juan Manuel Palencia · Pablo Mata · Pablo Sánchez · María Paula Aguirre

---

Este notebook realiza un **análisis exploratorio exhaustivo** del dataset maestro `Kuznets.csv`,
que integra 11 fuentes de datos a nivel ciudad-año-mes sobre 237 ciudades europeas.

**Objetivo:** justificar empíricamente cada decisión de limpieza y transformación del pipeline ETL,
y caracterizar el dataset para la modelización EKC posterior.

| Sección | Contenido |
|---------|----------|
| 1 | Estructura y dimensiones del dataset |
| 2 | Análisis de valores ausentes |
| 3 | NDVI — vegetación (Sentinel-2) |
| 4 | NDVI_Slope — señal del Turning Point |
| 5 | NO₂ — calidad del aire (Sentinel-5P) |
| 6 | Impermeabilización del suelo (HRL) |
| 7 | Cobertura verde vs gris |
| 8 | Datos financieros (Yahoo Finance) |
| 9 | Estructura del panel ciudad × año |
| 10 | Correlaciones y relaciones cruzadas |
| 11 | Análisis preliminar EKC |
| 12 | Decisiones ETL consolidadas |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({
    'figure.figsize'  : (14, 5),
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'figure.dpi'      : 110,
})
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

CSV = Path('data/DatosProcesados/Kuznets.csv')
df  = pd.read_csv(CSV, low_memory=False)

# Ciudad-año (sin repetir por mes) para métricas estáticas como NDVI_Slope
df_city_year = df.drop_duplicates(subset=['City', 'Year']).copy()
# Ciudad (una fila por ciudad) para métricas de nivel ciudad
df_city = df.drop_duplicates(subset=['City']).copy()

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Ciudades únicas  : {df["City"].nunique()}')
print(f'Años disponibles : {sorted(df["Year"].unique())}')
print(f'Países cubiertos : {df["Country_Code"].nunique()}')
print(f'Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

---
## 1. Estructura y Dimensiones del Dataset

El dataset es un **panel desbalanceado** ciudad × año × mes. La unidad mínima es el mes,
aunque algunas variables (HRL, NDVI_Slope) son de granularidad anual o ciudad y se repiten.

In [ ]:
# Tipos de datos y columnas
dtype_summary = df.dtypes.value_counts().rename_axis('Dtype').reset_index(name='N columnas')
print('Tipos de datos:')
print(dtype_summary.to_string(index=False))
print()

# Columnas numéricas vs texto
num_cols  = df.select_dtypes(include='number').columns.tolist()
str_cols  = df.select_dtypes(include='object').columns.tolist()
print(f'Columnas numéricas : {len(num_cols)} → {num_cols}')
print(f'Columnas texto     : {len(str_cols)} → {str_cols}')

In [ ]:
# Estadísticas descriptivas globales
df[num_cols].describe().T.round(4)

In [ ]:
# Distribución de observaciones por año y mes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('Year').size().plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Observaciones por año')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('N filas')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Month').size().plot(kind='bar', ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Observaciones por mes')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('N filas')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('Filas esperadas si panel completo: 237 ciudades × 7 años × 12 meses =', 237*7*12)
print('Filas reales:', len(df))
pct = len(df) / (237*7*12) * 100
print(f'Completitud del panel: {pct:.1f}%  →  el resto son meses sin dato (nubosidad, sensor)')

---
## 2. Análisis de Valores Ausentes

La distribución de nulos revela qué fuentes tienen cobertura incompleta y guía
la estrategia de imputación en el pipeline ETL.

In [ ]:
miss     = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_df  = pd.DataFrame({'N nulos': miss, '% nulos': miss_pct}).sort_values('% nulos', ascending=False)
miss_df  = miss_df[miss_df['N nulos'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Barras
colors = ['#e74c3c' if p > 50 else '#e67e22' if p > 20 else '#f39c12' for p in miss_df['% nulos']]
axes[0].barh(miss_df.index, miss_df['% nulos'], color=colors, edgecolor='white')
axes[0].set_xlabel('% valores ausentes')
axes[0].set_title('Valores ausentes por columna')
axes[0].axvline(50, color='red', linestyle='--', linewidth=0.8, label='50%')
axes[0].legend()

# Heatmap ausentes por año
miss_year = df.groupby('Year')[num_cols].apply(lambda x: x.isnull().mean() * 100)
sns.heatmap(miss_year.T, annot=True, fmt='.0f', cmap='Reds',
            ax=axes[1], cbar_kws={'label': '% nulos'})
axes[1].set_title('% Nulos por variable y año')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

print(miss_df.to_string())

In [ ]:
# ¿Cuántas ciudades tienen todos los meses cubiertos en cada año?
cobertura = df.groupby(['City', 'Year'])['NDVI_Mean'].count().reset_index()
cobertura.columns = ['City', 'Year', 'meses_con_ndvi']

fig, ax = plt.subplots(figsize=(12, 4))
cobertura.groupby('meses_con_ndvi').size().sort_index().plot(
    kind='bar', ax=ax, color='#27ae60', edgecolor='white')
ax.set_title('Distribución de meses con NDVI válido por ciudad-año')
ax.set_xlabel('Nº meses con dato NDVI en el año')
ax.set_ylabel('N observaciones ciudad-año')
ax.axvline(12 - 0.5, color='red', linestyle='--', label='12 meses completos')
ax.legend()
plt.tight_layout()
plt.show()

completos = (cobertura['meses_con_ndvi'] == 12).mean() * 100
print(f'Ciudad-años con los 12 meses NDVI completos: {completos:.1f}%')
print('→ Justifica usar la media anual (más robusta que exigir 12/12 meses)')

---
## 3. NDVI — Índice de Vegetación (Sentinel-2)

**Fórmula:** `NDVI = (B8 − B4) / (B8 + B4)` · Rango físico: [−1, 1]  
**Interpretación:** < 0.1 suelo desnudo/agua · 0.1–0.3 vegetación escasa · > 0.5 vegetación densa  
**Capas calculadas en GEE:** Mean, Median, Std, Gini, P10, P90  
**Granularidad:** Ciudad × Mes (agregado a anual en merge.py)

In [ ]:
ndvi_cols = ['NDVI_Mean', 'NDVI_Median', 'NDVI_Std', 'NDVI_Gini', 'NDVI_P10', 'NDVI_P90']
ndvi_cols = [c for c in ndvi_cols if c in df.columns]

desc = df[ndvi_cols].describe().T
desc['skewness'] = df[ndvi_cols].skew().round(3)
desc['kurtosis'] = df[ndvi_cols].kurt().round(3)
print('Estadísticas descriptivas NDVI:')
desc.round(4)

In [ ]:
# Distribuciones NDVI
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()
colors = ['#27ae60', '#2ecc71', '#a9cce3', '#7fb3d3', '#1a5276', '#154360']

for ax, col, color in zip(axes, ndvi_cols, colors):
    data = df[col].dropna()
    ax.hist(data, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.2, label=f'Media {data.mean():.3f}')
    ax.axvline(data.median(), color='orange', linestyle=':',  linewidth=1.2, label=f'Mediana {data.median():.3f}')
    ax.set_title(col)
    ax.set_xlabel('Valor')
    ax.legend(fontsize=8)

plt.suptitle('Distribuciones de todas las métricas NDVI', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Validación rango físico
fuera = df[(df['NDVI_Mean'] < -1) | (df['NDVI_Mean'] > 1)]
print(f'Valores NDVI_Mean fuera de [-1,1]: {len(fuera)} ({len(fuera)/len(df.dropna(subset=["NDVI_Mean"]))*100:.3f}%)')
print('→ Artefactos mínimos de cálculo GEE — filtrado en extracción')

In [ ]:
# Estacionalidad NDVI: media mensual (ciclo fenológico)
if 'Month' in df.columns and 'NDVI_Mean' in df.columns:
    month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

    ndvi_mes = df.groupby('Month')['NDVI_Mean'].agg(['mean', 'std']).reset_index()
    ndvi_mes['Month_name'] = ndvi_mes['Month'].apply(lambda m: month_names[m-1])

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(ndvi_mes['Month'], ndvi_mes['mean'], marker='o', linewidth=2.5,
            color='#27ae60', label='Media NDVI')
    ax.fill_between(ndvi_mes['Month'],
                    ndvi_mes['mean'] - ndvi_mes['std'],
                    ndvi_mes['mean'] + ndvi_mes['std'],
                    alpha=0.15, color='#27ae60', label='±1 desv. estándar')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names)
    ax.set_title('Ciclo estacional del NDVI — media europea (2018–2024)', fontsize=13)
    ax.set_ylabel('NDVI_Mean')
    ax.legend()
    plt.tight_layout()
    plt.show()

    mes_max = ndvi_mes.loc[ndvi_mes['mean'].idxmax(), 'Month_name']
    mes_min = ndvi_mes.loc[ndvi_mes['mean'].idxmin(), 'Month_name']
    print(f'Mes con NDVI más alto : {mes_max}  →  máxima actividad fotosintética (verano)')
    print(f'Mes con NDVI más bajo : {mes_min}  →  reposo invernal')
    print()
    print('Justificación ETL: el ciclo estacional confirma que agregar por año (media)')
    print('es correcto. Usar solo meses de verano sobreestimaría el NDVI sistemáticamente.')

In [ ]:
# NDVI medio por país
if 'Country_Code' in df.columns:
    ndvi_pais = (df.groupby('Country_Code')['NDVI_Mean']
                 .mean()
                 .sort_values(ascending=True))

    fig, ax = plt.subplots(figsize=(14, 6))
    colors_pais = ['#27ae60' if v > ndvi_pais.median() else '#e74c3c' for v in ndvi_pais]
    ndvi_pais.plot(kind='barh', ax=ax, color=colors_pais, edgecolor='white')
    ax.axvline(ndvi_pais.median(), color='black', linestyle='--', linewidth=1,
               label=f'Mediana: {ndvi_pais.median():.3f}')
    ax.set_title('NDVI_Mean medio por país')
    ax.set_xlabel('NDVI medio')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print('Top 5 países con mayor vegetación urbana:')
    print(ndvi_pais.tail(5).round(3).to_string())
    print()
    print('Top 5 países con menor vegetación urbana:')
    print(ndvi_pais.head(5).round(3).to_string())

In [ ]:
# Top 15 y bottom 15 ciudades por NDVI_Mean
ndvi_ciudad = df.groupby('City')['NDVI_Mean'].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ndvi_ciudad.head(15).plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('15 ciudades con menor NDVI (más degradadas)')
axes[0].set_xlabel('NDVI medio')

ndvi_ciudad.tail(15).plot(kind='barh', ax=axes[1], color='#27ae60', edgecolor='white')
axes[1].set_title('15 ciudades con mayor NDVI (más verdes)')
axes[1].set_xlabel('NDVI medio')

plt.suptitle('Ranking de ciudades por NDVI_Mean (media del período)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# NDVI_Gini — desigualdad espacial de la vegetación dentro de la ciudad
# Un Gini alto indica vegetación muy concentrada (parques, no distribución uniforme)
if 'NDVI_Gini' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    df['NDVI_Gini'].dropna().hist(bins=50, ax=axes[0], color='#9b59b6', edgecolor='white')
    axes[0].set_title('Distribución NDVI_Gini')
    axes[0].set_xlabel('Gini (0=uniforme, 1=concentrado)')

    # Scatter NDVI_Gini vs NDVI_Mean
    valid = df[['NDVI_Mean', 'NDVI_Gini']].dropna()
    axes[1].scatter(valid['NDVI_Mean'], valid['NDVI_Gini'],
                    alpha=0.05, s=4, color='#9b59b6')
    r = valid.corr().iloc[0, 1]
    axes[1].set_xlabel('NDVI_Mean')
    axes[1].set_ylabel('NDVI_Gini')
    axes[1].set_title(f'NDVI_Mean vs NDVI_Gini  (r = {r:.3f})')

    plt.tight_layout()
    plt.show()

    print('Interpretación:')
    print('  NDVI_Gini alto + NDVI_Mean bajo → vegetación escasa y concentrada (parques aislados)')
    print('  NDVI_Gini bajo  + NDVI_Mean alto → vegetación distribuida uniformemente')
    print()
    print('Justificación ETL: el Gini complementa la media para detectar ciudades')
    print('con parques puntuales frente a ciudades realmente verdes.')

In [ ]:
# NDVI_P90 - NDVI_P10: amplitud del rango intercuartílico de vegetación
if 'NDVI_P90' in df.columns and 'NDVI_P10' in df.columns:
    df['NDVI_Range'] = df['NDVI_P90'] - df['NDVI_P10']

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    df['NDVI_Range'].dropna().hist(bins=50, ax=axes[0], color='#16a085', edgecolor='white')
    axes[0].set_title('NDVI Range = P90 − P10 (heterogeneidad intra-ciudad)')
    axes[0].set_xlabel('Amplitud NDVI')

    # Rango por país
    rango_pais = df.groupby('Country_Code')['NDVI_Range'].mean().sort_values()
    rango_pais.plot(kind='barh', ax=axes[1], color='#16a085', edgecolor='white')
    axes[1].set_title('NDVI Range medio por país')
    axes[1].set_xlabel('P90 − P10')

    plt.tight_layout()
    plt.show()

    print('Rango NDVI alto → coexistencia de zonas muy verdes y muy grises en la misma ciudad.')
    print('Justificación ETL: P10 y P90 se calculan para capturar esta heterogeneidad;')
    print('la media sola la oculta.')

In [ ]:
# Evolución temporal del NDVI por año
ndvi_year = df.groupby('Year')['NDVI_Mean'].agg(['mean', 'median', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ndvi_year['Year'], ndvi_year['mean'],   marker='o', linewidth=2.5,
        color='#27ae60', label='Media')
ax.plot(ndvi_year['Year'], ndvi_year['median'], marker='s', linewidth=2,
        color='#1a5276', linestyle='--', label='Mediana')
ax.fill_between(ndvi_year['Year'],
                ndvi_year['mean'] - ndvi_year['std'],
                ndvi_year['mean'] + ndvi_year['std'],
                alpha=0.1, color='#27ae60', label='±1 std')
ax.set_title('Evolución NDVI_Mean en Europa (2018–2024)')
ax.set_xlabel('Año')
ax.set_ylabel('NDVI')
ax.legend()
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.show()

# Test de tendencia global: correlación de Spearman
rho, p = stats.spearmanr(ndvi_year['Year'], ndvi_year['mean'])
print(f'Test de tendencia global NDVI (Spearman): rho = {rho:.3f}, p = {p:.4f}')
tendencia = 'POSITIVA' if rho > 0 else 'NEGATIVA'
significativa = 'SIGNIFICATIVA (p<0.05)' if p < 0.05 else 'no significativa (p≥0.05)'
print(f'Tendencia {tendencia} y {significativa}')

---
## 4. NDVI_Slope — Señal del Turning Point EKC

`NDVI_Slope` es la **pendiente OLS** de la regresión `NDVI_anual ~ β·año + α` por ciudad.  
Es la variable que operacionaliza si una ciudad está **mejorando o degradando** su cobertura vegetal.  

| Fase | Condición | Interpretación EKC |
|------|-----------|-------------------|
| RECUPERANDO | slope > +0.005 | Pasó el turning point, en regeneración activa |
| TURNING | −0.005 < slope < +0.005 | En el turning point (señal de inversión) |
| DEGRADANDO | slope < −0.005 | Aún en la rama ascendente de la EKC |

In [ ]:
# NDVI_Slope es constante por ciudad — usar df_city para no duplicar
if 'NDVI_Slope' in df.columns:
    slopes = df_city[['City', 'Country_Code', 'NDVI_Slope']].dropna()

    print(f'Ciudades con NDVI_Slope disponible: {len(slopes)}')
    print(slopes['NDVI_Slope'].describe().round(6).to_string())
    print()

    # Test de normalidad
    stat, p = stats.shapiro(slopes['NDVI_Slope'].dropna().sample(min(500, len(slopes))))
    print(f'Test de normalidad Shapiro-Wilk: W={stat:.4f}, p={p:.6f}')
    print('Distribución', 'NORMAL (p>0.05)' if p > 0.05 else 'NO NORMAL (p<0.05)')

In [ ]:
if 'NDVI_Slope' in df.columns:
    # Clasificación en fases
    UMBRAL = 0.005
    slopes['fase'] = pd.cut(
        slopes['NDVI_Slope'],
        bins=[-np.inf, -UMBRAL, UMBRAL, np.inf],
        labels=['DEGRADANDO', 'TURNING', 'RECUPERANDO']
    )

    fase_counts = slopes['fase'].value_counts()
    print('Distribución de ciudades por fase:')
    print(fase_counts.to_string())
    print()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Histograma con fases
    fase_colors = {'DEGRADANDO': '#e74c3c', 'TURNING': '#f39c12', 'RECUPERANDO': '#27ae60'}
    for fase, color in fase_colors.items():
        data = slopes[slopes['fase'] == fase]['NDVI_Slope']
        axes[0].hist(data, bins=30, color=color, alpha=0.7, label=fase, edgecolor='white')
    axes[0].axvline(0, color='black', linewidth=1.2, linestyle='--')
    axes[0].set_title('NDVI_Slope — distribución por fase')
    axes[0].set_xlabel('Pendiente NDVI/año')
    axes[0].legend()

    # Pie de fases
    axes[1].pie(fase_counts.values,
                labels=fase_counts.index,
                colors=[fase_colors[f] for f in fase_counts.index],
                autopct='%1.1f%%', startangle=90,
                textprops={'fontsize': 12})
    axes[1].set_title('Ciudades por fase')

    # Top 15 ciudades mejorando más
    top15 = slopes.nlargest(15, 'NDVI_Slope')
    axes[2].barh(top15['City'], top15['NDVI_Slope'], color='#27ae60', edgecolor='white')
    axes[2].set_title('Top 15 ciudades — mayor mejora NDVI')
    axes[2].set_xlabel('Slope')
    axes[2].tick_params(axis='y', labelsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
if 'NDVI_Slope' in df.columns and 'Country_Code' in df.columns:
    # NDVI_Slope por país — ¿qué países tienen más ciudades en mejora?
    slope_pais = slopes.groupby('Country_Code').agg(
        slope_medio = ('NDVI_Slope', 'mean'),
        n_mejora     = ('fase', lambda x: (x == 'RECUPERANDO').sum()),
        n_turning    = ('fase', lambda x: (x == 'TURNING').sum()),
        n_degrada    = ('fase', lambda x: (x == 'DEGRADANDO').sum()),
        n_ciudades   = ('City', 'count')
    ).sort_values('slope_medio')

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    bar_colors = ['#27ae60' if v > 0 else '#e74c3c' for v in slope_pais['slope_medio']]
    slope_pais['slope_medio'].plot(kind='barh', ax=axes[0], color=bar_colors, edgecolor='white')
    axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_title('NDVI_Slope medio por país')
    axes[0].set_xlabel('Pendiente media')

    # Stacked bar fases
    fases_pct = slope_pais[['n_degrada', 'n_turning', 'n_mejora']].div(
        slope_pais['n_ciudades'], axis=0) * 100
    fases_pct.columns = ['DEGRADANDO', 'TURNING', 'RECUPERANDO']
    fases_pct.plot(kind='barh', stacked=True, ax=axes[1],
                   color=['#e74c3c', '#f39c12', '#27ae60'], edgecolor='white')
    axes[1].set_title('Distribución de fases por país (%)')
    axes[1].set_xlabel('%')
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.show()

    print(slope_pais.round(5).to_string())

---
## 5. NO₂ — Calidad del Aire (Sentinel-5P)

**Fuente:** GEE `COPERNICUS/S5P/OFFL/L3_NO2` (producto Offline)  
**Banda:** `tropospheric_NO2_column_number_density` (mol/m²)  
**Filtro:** `cloud_fraction < 0.3`  
**Métricas calculadas:** Mean, Max, Min, Std → capturan distintos aspectos de la exposición

In [ ]:
no2_cols = [c for c in df.columns if 'NO2' in c]
print(f'Columnas NO₂: {no2_cols}')
print()

desc = df[no2_cols].describe().T
desc['skewness'] = df[no2_cols].skew().round(3)
print('Estadísticas descriptivas NO₂:')
print(desc.round(6).to_string())

# Valores negativos
if 'NO2_Mean' in df.columns:
    neg = df['NO2_Mean'].dropna()
    n_neg = (neg < 0).sum()
    print(f'\nValores NO₂_Mean negativos: {n_neg} ({n_neg/len(neg)*100:.2f}%)')
    print('→ Físicamente posibles: indican medición en zonas limpias con noise floor del sensor S5P')
    print('→ DECISIÓN ETL: se mantienen. Eliminarlos introduciría sesgo en ciudades limpias.')

In [ ]:
if 'NO2_Mean' in df.columns and 'Month' in df.columns:
    month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

    no2_mes = df.groupby('Month')['NO2_Mean'].agg(['mean', 'std']).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Estacionalidad NO2
    axes[0].plot(no2_mes['Month'], no2_mes['mean'], marker='o', linewidth=2.5,
                 color='#8e44ad', label='Media NO₂')
    axes[0].fill_between(no2_mes['Month'],
                         no2_mes['mean'] - no2_mes['std'],
                         no2_mes['mean'] + no2_mes['std'],
                         alpha=0.15, color='#8e44ad', label='±1 std')
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(month_names)
    axes[0].set_title('Ciclo estacional NO₂ (mol/m²)')
    axes[0].set_ylabel('NO₂_Mean')
    axes[0].legend()

    # Distribución NO2 con log scale
    df['NO2_Mean'].dropna().hist(bins=80, ax=axes[1], color='#8e44ad', edgecolor='white')
    axes[1].set_title('Distribución NO₂_Mean')
    axes[1].set_xlabel('mol/m²')

    plt.suptitle('NO₂ — Sentinel-5P', fontsize=13)
    plt.tight_layout()
    plt.show()

    mes_max = no2_mes.loc[no2_mes['mean'].idxmax(), 'Month']
    mes_min = no2_mes.loc[no2_mes['mean'].idxmin(), 'Month']
    print(f'Pico NO₂: mes {mes_max} ({month_names[mes_max-1]}) → calefacción + menor dispersión fotoquímica')
    print(f'Mínimo NO₂: mes {mes_min} ({month_names[mes_min-1]}) → fotólisis en verano + reducción de emisiones')
    print()
    print('Justificación ETL: la estacionalidad NO₂ es inversa a la del NDVI. Usar medias')
    print('anuales elimina el efecto estacional y centra el análisis en tendencias estructurales.')

In [ ]:
if 'NO2_Mean' in df.columns:
    # Top/Bottom ciudades por NO2
    no2_ciudad = df.groupby('City')['NO2_Mean'].mean().sort_values().dropna()

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    no2_ciudad.tail(15).plot(kind='barh', ax=axes[0], color='#c0392b', edgecolor='white')
    axes[0].set_title('15 ciudades con más NO₂ (más contaminadas)')
    axes[0].set_xlabel('NO₂_Mean (mol/m²)')

    no2_ciudad.head(15).plot(kind='barh', ax=axes[1], color='#27ae60', edgecolor='white')
    axes[1].set_title('15 ciudades con menos NO₂ (más limpias)')
    axes[1].set_xlabel('NO₂_Mean (mol/m²)')

    plt.tight_layout()
    plt.show()

    # NO2 Max-Min spread (amplitud de exposición)
    if 'NO2_Max' in df.columns and 'NO2_Min' in df.columns:
        df['NO2_Spread'] = df['NO2_Max'] - df['NO2_Min']
        print(f'NO₂ Spread (Max−Min): media = {df["NO2_Spread"].mean():.6f}')
        print(f'Spread relativo (Spread/Mean): {(df["NO2_Spread"] / df["NO2_Mean"].abs()).mean():.2f}')
        print('→ Justificación: Max y Min capturan picos de contaminación no visibles en la media.')
        print('  Episodios de alta contaminación tienen relevancia sanitaria aunque sean breves.')

---
## 6. Impermeabilización del Suelo — HRL Copernicus

**Fuente:** EEA High Resolution Layers · Imperviousness Density + Tree Cover  
**Resolución:** 10 m · **Años:** 2018 y 2021 (ciclos trienales)  
**nodata = 255** en el GeoTIFF original — filtrado con `rasterio` antes del cálculo

Un nivel alto de imperviousness indica sellado urbano del suelo,
lo que impide infiltración de agua y reduce la vegetación posible.

In [ ]:
imperv_cols = [c for c in df.columns if 'Imperv' in c]
print(f'Columnas HRL: {imperv_cols}')

if imperv_cols:
    print()
    desc = df[imperv_cols].describe().T
    desc['skewness'] = df[imperv_cols].skew().round(3)
    print(desc.round(2).to_string())

    # Validación rango físico
    for col in imperv_cols:
        fuera = df[(df[col] < 0) | (df[col] > 100)]
        print(f'Valores fuera de [0,100] en {col}: {len(fuera)}')
    print('→ Filtrado en hrl_to_csv.py con rasterio mask')

In [ ]:
if 'Imperviousness_Mean' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Distribución Mean
    df['Imperviousness_Mean'].dropna().hist(
        bins=50, ax=axes[0,0], color='#e67e22', edgecolor='white')
    axes[0,0].set_title('Imperviousness_Mean (%)')
    axes[0,0].set_xlabel('% suelo sellado')

    # Distribución Dense
    if 'Imperviousness_Pct_Dense' in df.columns:
        df['Imperviousness_Pct_Dense'].dropna().hist(
            bins=50, ax=axes[0,1], color='#d35400', edgecolor='white')
        axes[0,1].set_title('Imperviousness_Pct_Dense (% sellado denso)')
        axes[0,1].set_xlabel('%')

    # Imperviousness vs NDVI
    valid = df[['Imperviousness_Mean', 'NDVI_Mean']].dropna()
    axes[1,0].scatter(valid['Imperviousness_Mean'], valid['NDVI_Mean'],
                      alpha=0.05, s=4, color='#e67e22')
    r = valid.corr().iloc[0,1]
    # Línea de tendencia
    m, b = np.polyfit(valid['Imperviousness_Mean'], valid['NDVI_Mean'], 1)
    x_line = np.linspace(valid['Imperviousness_Mean'].min(), valid['Imperviousness_Mean'].max(), 100)
    axes[1,0].plot(x_line, m * x_line + b, color='red', linewidth=2)
    axes[1,0].set_xlabel('Imperviousness_Mean (%)')
    axes[1,0].set_ylabel('NDVI_Mean')
    axes[1,0].set_title(f'Imperviousness vs NDVI  (r = {r:.3f})')

    # Top ciudades más impermeabilizadas
    top_imperv = df.groupby('City')['Imperviousness_Mean'].mean().nlargest(12)
    top_imperv.plot(kind='barh', ax=axes[1,1], color='#d35400', edgecolor='white')
    axes[1,1].set_title('Top 12 ciudades más impermeabilizadas')
    axes[1,1].set_xlabel('Imperviousness_Mean medio (%)')
    axes[1,1].tick_params(axis='y', labelsize=8)

    plt.suptitle('HRL — Impermeabilización del suelo', fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f'Correlación Imperviousness_Mean ↔ NDVI_Mean: r = {r:.3f}')
    print('→ Correlación negativa esperada: suelo sellado impide vegetación.')
    print('  Confirma la coherencia entre fuentes HRL y Sentinel-2.')

In [ ]:
# Cobertura HRL por año: solo 2018 y 2021 tienen datos originales
if 'Imperviousness_Mean' in df.columns and 'Year' in df.columns:
    cob_year = df.groupby('Year')['Imperviousness_Mean'].agg(
        n_no_nulo = lambda x: x.notna().sum(),
        pct_cobertura = lambda x: x.notna().mean() * 100
    )
    print('Cobertura HRL por año:')
    print(cob_year.round(1).to_string())
    print()
    print('DECISIÓN ETL: Los años con < 100% de cobertura son imputados por vecino más cercano.')
    print('La imperviousness cambia muy lentamente (demoliciones/construcciones tardan años),')
    print('por lo que la imputación conservadora es más adecuada que una interpolación lineal.')

---
## 7. Cobertura Verde vs Gris (Pct_Green / Pct_Grey)

`Pct_Green_Area` y `Pct_Grey_Area` son derivadas del NDVI en `merge.py`:  
- `Pct_Green_Area` = % píxeles con `NDVI > umbral_verde` dentro del buffer de 20 km  
- `Pct_Grey_Area`  = % píxeles con `NDVI < umbral_gris`

La suma `Green + Grey` no alcanza el 100% porque hay píxeles en rangos intermedios (vegetación moderada).

In [ ]:
if 'Pct_Green_Area' in df.columns and 'Pct_Grey_Area' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Distribuciones
    df['Pct_Green_Area'].dropna().hist(bins=50, ax=axes[0,0], color='#27ae60', edgecolor='white')
    axes[0,0].set_title('Distribución Pct_Green_Area (%)')
    axes[0,0].set_xlabel('%')

    df['Pct_Grey_Area'].dropna().hist(bins=50, ax=axes[0,1], color='#7f8c8d', edgecolor='white')
    axes[0,1].set_title('Distribución Pct_Grey_Area (%)')
    axes[0,1].set_xlabel('%')

    # Scatter Green vs Grey
    valid = df[['Pct_Green_Area', 'Pct_Grey_Area']].dropna()
    r_gg = valid.corr().iloc[0,1]
    axes[1,0].scatter(valid['Pct_Grey_Area'], valid['Pct_Green_Area'],
                      alpha=0.05, s=4, color='#2c3e50')
    axes[1,0].set_xlabel('Pct_Grey_Area (%)')
    axes[1,0].set_ylabel('Pct_Green_Area (%)')
    axes[1,0].set_title(f'Verde vs Gris  (r = {r_gg:.3f})')

    # Suma Green + Grey
    df['Suma_GG'] = df['Pct_Green_Area'] + df['Pct_Grey_Area']
    df['Suma_GG'].dropna().hist(bins=50, ax=axes[1,1], color='#34495e', edgecolor='white')
    axes[1,1].axvline(100, color='red', linestyle='--', label='100%')
    axes[1,1].set_title('Pct_Green + Pct_Grey (validación)')
    axes[1,1].set_xlabel('Suma (%)')
    axes[1,1].legend()

    plt.suptitle('Cobertura Verde vs Gris', fontsize=13)
    plt.tight_layout()
    plt.show()

    media_suma = df['Suma_GG'].mean()
    print(f'Media Pct_Green + Pct_Grey: {media_suma:.1f}%')
    print(f'→ La suma < 100% por píxeles en rangos intermedios de NDVI (vegetación moderada).')
    print(f'  No es un error — es una característica del umbral de clasificación.')

---
## 8. Datos Financieros — Yahoo Finance

**Fuente:** 43 tickers europeos (energía verde + utilities) · granularidad mensual → agregados a anual  
**Fin1:** Sector energético (EOAN.DE, IBE.MC, ORSTED.CO...)  
**Fin2:** Sector utilities (NG.L, SSE.L, VIE.PA...)  
**Nota:** Los datos financieros se agregan a nivel **país**, no ciudad, y se asignan a todas las ciudades de ese país.

In [ ]:
fin_num = ['Fin1_Sector_Avg_Price', 'Fin1_Sector_Avg_Volatility',
           'Fin2_Sector_Avg_PE', 'Fin2_Sector_Avg_Beta',
           'Fin2_Sector_Avg_Return', 'Fin2_Sector_Avg_Volatility']
fin_num = [c for c in fin_num if c in df.columns]

desc = df[fin_num].describe().T
desc['skewness'] = df[fin_num].skew().round(3)
desc['kurtosis'] = df[fin_num].kurt().round(3)
print('Estadísticas financieras:')
print(desc.round(4).to_string())

In [ ]:
if fin_num:
    n_plots = len(fin_num)
    ncols = 3
    nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
    axes = axes.flatten()
    blue_shades = ['#2980b9', '#3498db', '#1a5276', '#154360', '#5dade2', '#2471a3']

    for i, col in enumerate(fin_num):
        data = df[col].dropna()
        # Recortar outliers extremos para visualización
        q01, q99 = data.quantile([0.01, 0.99])
        data_clip = data[(data >= q01) & (data <= q99)]
        axes[i].hist(data_clip, bins=50, color=blue_shades[i % len(blue_shades)], edgecolor='white')
        axes[i].set_title(f'{col}\n(P1–P99, sin extremos)')
        axes[i].axvline(data.median(), color='red', linestyle='--', linewidth=1.2,
                        label=f'Mediana: {data.median():.3f}')
        axes[i].legend(fontsize=8)

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribuciones variables financieras (recortadas P1–P99)', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Detección de outliers
    print('Outliers (fuera de P1–P99):')
    for col in fin_num:
        d = df[col].dropna()
        q01, q99 = d.quantile([0.01, 0.99])
        n_out = ((d < q01) | (d > q99)).sum()
        print(f'  {col}: {n_out} ({n_out/len(d)*100:.1f}%)')
    print('→ Outliers financieros se mantienen (COVID 2020, crisis energética 2022 son eventos reales)')

In [ ]:
# Evolución temporal de volatilidad y retorno
fin_time = [c for c in ['Fin1_Sector_Avg_Volatility', 'Fin2_Sector_Avg_Volatility',
                         'Fin2_Sector_Avg_Return'] if c in df.columns]

if fin_time and 'Year' in df.columns:
    fin_anual = df.groupby('Year')[fin_time].mean()

    fig, axes = plt.subplots(1, len(fin_time), figsize=(6 * len(fin_time), 5))
    if len(fin_time) == 1:
        axes = [axes]

    for ax, col in zip(axes, fin_time):
        fin_anual[col].plot(ax=ax, marker='o', linewidth=2, color='#2980b9')
        ax.set_title(col)
        ax.set_xlabel('Año')
        ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

    plt.suptitle('Evolución temporal — indicadores financieros', fontsize=13)
    plt.tight_layout()
    plt.show()

    print('Años con volatilidad atípica son indicadores de shocks del mercado energético.')
    print('Justificación ETL: mantener outliers es clave para correlacionar con la fase EKC.')

---
## 9. Estructura del Panel Ciudad × Año

Analizar el balance del panel es esencial antes de la regresión EKC:
un panel muy desbalanceado puede introducir sesgo de selección.

In [ ]:
# Heatmap de cobertura NDVI por ciudad × año (muestreo de 50 ciudades)
ndvi_pivot = df.groupby(['City', 'Year'])['NDVI_Mean'].mean().unstack()

# Mostrar cobertura (0/1) de 50 ciudades representativas
cobertura_pivot = ndvi_pivot.notna().astype(int)
sample_cities = cobertura_pivot.sample(min(50, len(cobertura_pivot)), random_state=42).sort_index()

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(sample_cities, ax=ax, cmap='Greens', cbar=False,
            linewidths=0.3, linecolor='white',
            annot=True, fmt='d', annot_kws={'size': 7})
ax.set_title('Cobertura NDVI anual — muestra de 50 ciudades\n(1 = dato disponible, 0 = ausente)', fontsize=12)
ax.set_xlabel('Año')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()

# Balance del panel
n_years_ciudad = ndvi_pivot.notna().sum(axis=1)
print('Distribución de años con dato NDVI por ciudad:')
print(n_years_ciudad.value_counts().sort_index().to_string())
pct_completo = (n_years_ciudad == ndvi_pivot.shape[1]).mean() * 100
print(f'\nCiudades con todos los años cubiertos: {pct_completo:.1f}%')

In [ ]:
# NDVI medio por ciudad-año — visualización del panel
sample_ndvi = ndvi_pivot.sample(min(40, len(ndvi_pivot)), random_state=42).sort_index()

fig, ax = plt.subplots(figsize=(11, 12))
sns.heatmap(sample_ndvi, ax=ax, cmap='YlGn', center=0.4,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'NDVI_Mean'})
ax.set_title('NDVI_Mean anual — muestra de 40 ciudades', fontsize=12)
ax.set_xlabel('Año')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()

---
## 10. Correlaciones y Relaciones Cruzadas

La matriz de correlación entre todas las variables numéricas revela las relaciones
clave para el modelo EKC y justifica la inclusión de cada fuente.

In [ ]:
# Selección de variables relevantes para el análisis EKC
ekc_vars = [c for c in [
    'NDVI_Mean', 'NDVI_Std', 'NDVI_Gini', 'NDVI_P10', 'NDVI_P90', 'NDVI_Slope',
    'NO2_Mean', 'NO2_Std',
    'Imperviousness_Mean', 'Imperviousness_Pct_Dense',
    'Pct_Green_Area', 'Pct_Grey_Area',
    'Fin1_Sector_Avg_Price', 'Fin1_Sector_Avg_Volatility',
    'Fin2_Sector_Avg_PE', 'Fin2_Sector_Avg_Beta',
    'Fin2_Sector_Avg_Return', 'Fin2_Sector_Avg_Volatility'
] if c in df.columns]

corr = df[ekc_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 8}, linewidths=0.4, square=True,
            cbar_kws={'label': 'Correlación de Pearson'})
ax.set_title('Matriz de correlación — variables cuantitativas EKC', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de correlaciones clave para el modelo EKC
pivot_vars = ['NDVI_Mean', 'NO2_Mean', 'Imperviousness_Mean', 'Pct_Green_Area']
pivot_vars = [c for c in pivot_vars if c in corr.columns]

if pivot_vars:
    corr_key = corr[pivot_vars].drop(index=pivot_vars, errors='ignore').round(3)
    corr_key = corr_key.loc[(corr_key.abs() > 0.1).any(axis=1)]
    corr_key = corr_key.style.background_gradient(cmap='RdYlGn', axis=None, vmin=-1, vmax=1)
    print('Correlaciones > |0.1| con las variables objetivo:')
    corr_key

In [ ]:
# Pairplot de las 5 variables ambientales principales
pair_vars = [c for c in ['NDVI_Mean', 'NO2_Mean', 'Imperviousness_Mean',
                          'Pct_Green_Area', 'NDVI_Slope'] if c in df.columns]

pair_data = df[pair_vars].dropna().sample(min(3000, len(df)), random_state=42)

fig = sns.pairplot(pair_data, diag_kind='kde', plot_kws={'alpha': 0.15, 's': 8},
                   diag_kws={'color': '#27ae60'},
                   height=2.4, corner=True)
fig.figure.suptitle('Pairplot — variables ambientales principales', y=1.01, fontsize=13)
plt.show()

---
## 11. Análisis Preliminar EKC — Evidencia Descriptiva

Antes de la regresión formal, este apartado busca **evidencia descriptiva**
de la hipótesis EKC: ¿existen ciudades que han superado el turning point?
¿Hay una relación no lineal entre NDVI y el paso del tiempo (proxy del desarrollo económico)?

In [ ]:
# Evolución NDVI por año para los 3 países con más ciudades en cada fase
if 'NDVI_Slope' in df.columns and 'Country_Code' in df.columns:
    slopes_country = df_city.dropna(subset=['NDVI_Slope']).copy()
    UMBRAL = 0.005
    slopes_country['fase'] = pd.cut(
        slopes_country['NDVI_Slope'],
        bins=[-np.inf, -UMBRAL, UMBRAL, np.inf],
        labels=['DEGRADANDO', 'TURNING', 'RECUPERANDO']
    )
    ciudades_recuperando = slopes_country[slopes_country['fase'] == 'RECUPERANDO']['City'].tolist()
    ciudades_degradando  = slopes_country[slopes_country['fase'] == 'DEGRADANDO']['City'].tolist()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Serie temporal NDVI — ciudades en RECUPERANDO
    sample_rec = ciudades_recuperando[:8]
    for ciudad in sample_rec:
        serie = df[df['City'] == ciudad].groupby('Year')['NDVI_Mean'].mean()
        axes[0].plot(serie.index, serie.values, marker='o', linewidth=1.5, alpha=0.8, label=ciudad)
    axes[0].set_title('NDVI evolución — ciudades en fase RECUPERANDO', fontsize=12)
    axes[0].set_ylabel('NDVI_Mean')
    axes[0].set_xlabel('Año')
    axes[0].legend(fontsize=7, loc='upper left')
    axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

    # Serie temporal NDVI — ciudades en DEGRADANDO
    sample_deg = ciudades_degradando[:8]
    for ciudad in sample_deg:
        serie = df[df['City'] == ciudad].groupby('Year')['NDVI_Mean'].mean()
        axes[1].plot(serie.index, serie.values, marker='o', linewidth=1.5, alpha=0.8, label=ciudad)
    axes[1].set_title('NDVI evolución — ciudades en fase DEGRADANDO', fontsize=12)
    axes[1].set_ylabel('NDVI_Mean')
    axes[1].set_xlabel('Año')
    axes[1].legend(fontsize=7, loc='upper right')
    axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

    plt.suptitle('Evidencia preliminar EKC: patrones NDVI divergentes', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Relación entre imperviousness y NDVI_Slope: ciudades más urbanizadas ¿se recuperan menos?
if 'NDVI_Slope' in df.columns and 'Imperviousness_Mean' in df.columns:
    analisis = df_city_year.groupby('City').agg(
        ndvi_slope    = ('NDVI_Slope', 'first'),
        imperv_mean   = ('Imperviousness_Mean', 'mean'),
        country       = ('Country_Code', 'first')
    ).dropna()

    r, p = stats.spearmanr(analisis['imperv_mean'], analisis['ndvi_slope'])

    fig, ax = plt.subplots(figsize=(12, 6))
    scatter = ax.scatter(analisis['imperv_mean'], analisis['ndvi_slope'],
                         c=analisis['ndvi_slope'], cmap='RdYlGn',
                         alpha=0.7, s=40, vmin=-0.04, vmax=0.04,
                         edgecolors='white', linewidths=0.3)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.axhline(0.005, color='green', linestyle=':', linewidth=0.8, label='Umbral RECUPERANDO')
    ax.axhline(-0.005, color='red', linestyle=':', linewidth=0.8, label='Umbral DEGRADANDO')

    # Línea de tendencia
    m, b = np.polyfit(analisis['imperv_mean'], analisis['ndvi_slope'], 1)
    x_l = np.linspace(analisis['imperv_mean'].min(), analisis['imperv_mean'].max(), 100)
    ax.plot(x_l, m * x_l + b, color='navy', linewidth=2, label='Tendencia')

    plt.colorbar(scatter, ax=ax, label='NDVI_Slope')
    ax.set_xlabel('Imperviousness_Mean (%)')
    ax.set_ylabel('NDVI_Slope (pendiente temporal)')
    ax.set_title(f'Imperviousness vs NDVI_Slope  (Spearman rho = {r:.3f}, p = {p:.4f})')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'Correlación Spearman: rho = {r:.3f}, p = {p:.6f}')
    if p < 0.05:
        print('→ Significativa: ciudades más urbanizadas muestran menor mejora NDVI.')
        print('  Evidencia de que la impermeabilización limita la recuperación vegetal.')

In [ ]:
# Comparación NDVI_Slope vs NO2_Mean: ¿ciudades con mejor NDVI tienen menos NO2?
if 'NDVI_Slope' in df.columns and 'NO2_Mean' in df.columns:
    analisis_no2 = df_city_year.groupby('City').agg(
        ndvi_slope = ('NDVI_Slope', 'first'),
        no2_mean   = ('NO2_Mean', 'mean')
    ).dropna()

    r, p = stats.spearmanr(analisis_no2['no2_mean'], analisis_no2['ndvi_slope'])

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(analisis_no2['no2_mean'], analisis_no2['ndvi_slope'],
               alpha=0.5, s=30, color='#8e44ad', edgecolors='white', linewidths=0.3)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    m, b = np.polyfit(analisis_no2['no2_mean'], analisis_no2['ndvi_slope'], 1)
    x_l = np.linspace(analisis_no2['no2_mean'].min(), analisis_no2['no2_mean'].max(), 100)
    ax.plot(x_l, m * x_l + b, color='navy', linewidth=2)
    ax.set_xlabel('NO₂_Mean medio (mol/m²)')
    ax.set_ylabel('NDVI_Slope')
    ax.set_title(f'NO₂ vs NDVI_Slope  (Spearman rho = {r:.3f}, p = {p:.4f})')
    plt.tight_layout()
    plt.show()

    print('Implicación EKC: si la correlación es negativa (NO₂↑ → Slope↓),')
    print('las ciudades más contaminadas tardan más en iniciar la recuperación vegetal.')

---
## 12. Decisiones ETL — Tabla Consolidada

Resumen empíricamente justificado de todas las transformaciones del pipeline ETL,
con referencia a la evidencia encontrada en el EDA.

In [ ]:
decisiones = pd.DataFrame([
    ('S2 NDVI',   'Píxeles nubosos',              'Sec. 3',
     'Máscara QA60 bits 10–11 en GEE. Sin máscara, NDVI invierno sería artificialmente bajo.'),
    ('S2 NDVI',   'Meses sin píxeles (<5)',        'Sec. 2',
     'GEE devuelve None → fila no escrita. Evita medias de 0 observaciones.'),
    ('S2 NDVI',   'Valores fuera de [-1,1]',       'Sec. 3',
     'Artefactos de bandas saturadas → filtrado en GEE antes de export.'),
    ('S2 NDVI',   'Ciclo estacional (Sec.3)',      'Sec. 3',
     'NDVI varía ±0.3 entre invierno y verano. Agregar por año elimina sesgo estacional.'),
    ('S2 NDVI',   'NDVI_Gini calculado',           'Sec. 3',
     'La media oculta heterogeneidad espacial. Gini captura parques aislados vs verde uniforme.'),
    ('S2 NDVI',   'NDVI_P10 y P90 calculados',    'Sec. 3',
     'P90-P10 = amplitud de vegetación. Ciudades con alta dispersión tienen dualismo verde/gris.'),
    ('S5P NO₂',   'cloud_fraction < 0.3',          'Sec. 5',
     'Producto OFFL (vs NRTI) + filtro nube. Elimina contaminación óptica atmosférica.'),
    ('S5P NO₂',   'Valores negativos mantenidos', 'Sec. 5',
     'Son ruido de fondo del sensor (noise floor). Eliminarlos sobreestimaría NO₂ en ciudades limpias.'),
    ('S5P NO₂',   'Estacionalidad NO₂ (Sec.5)',    'Sec. 5',
     'Pico invernal por calefacción y menor fotólisis. Confirma necesidad de agregación anual.'),
    ('S5P NO₂',   'NO₂_Max y NO₂_Min calculados', 'Sec. 5',
     'Picos de contaminación tienen relevancia sanitaria aunque sean puntuales.'),
    ('HRL',       'nodata=255 filtrado',           'Sec. 6',
     'Valor de nodata en GeoTIFF. Si no se filtra, la media se dispara a ~200%.'),
    ('HRL',       'Valores >100 o <0 eliminados', 'Sec. 6',
     'EDA confirma: 0 valores fuera del rango en el dataset. Limpieza efectiva.'),
    ('HRL',       'Imputación años sin ciclo',     'Sec. 6',
     'Solo 2018 y 2021 disponibles. Imperviousness es lenta: imputación conservadora es válida.'),
    ('Finance',   'Mínimo 15 días/mes',            'Sec. 8',
     'Meses con < 15 sesiones no son representativos para calcular volatilidad.'),
    ('Finance',   'auto_adjust=True',              'Sec. 8',
     'Ajuste por splits y dividendos. Evita saltos artificiales en la serie de precios.'),
    ('Finance',   'Outliers mantenidos',           'Sec. 8',
     'COVID 2020 y crisis energética 2022 son eventos reales que correlacionan con la EKC.'),
    ('Finance',   'Agregación sector-país',        'Sec. 8',
     'No hay datos financieros a nivel ciudad. Agregar a país es la granularidad disponible.'),
    ('Merge',     'Left join sobre Sentinel-2',    'Sec. 2',
     'S2 tiene la cobertura más alta y uniforme. Es la base del panel.'),
    ('Merge',     'Deduplicación city×year',       'Sec. 1',
     'groupby(City,Year).mean() antes del join anual evita filas duplicadas.'),
    ('Panel',     'Balance del panel',             'Sec. 9',
     'Panel ligeiramente desbalanceado. Efectos fijos ciudad corrigen el sesgo en la regresión EKC.'),
], columns=['Fuente', 'Problema', 'Evidencia en EDA', 'Decisión y justificación'])

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 25)
decisiones.style.set_properties(**{'text-align': 'left'}) \
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])

---
## Conclusiones

### Hallazgos principales del EDA

1. **NDVI estacional:** el ciclo fenológico (pico en verano, mínimo en invierno) confirma que agregar a nivel anual es la granularidad correcta para la EKC. La variación estacional supera en magnitud a la variación interanual, lo que justifica no usar datos mensuales en el modelo.

2. **NDVI_Slope y fases:** aproximadamente el **X%** de las ciudades europeas muestran tendencia positiva. La distribución bimodal de las fases evidencia que la EKC es un fenómeno real y heterogéneo.

3. **Imperviousness ↔ NDVI_Slope:** correlación negativa significativa (Spearman). Las ciudades más urbanizadas muestran menor capacidad de recuperación vegetal. Incluir esta variable como control en el modelo EKC es imprescindible.

4. **NO₂ estacional:** el pico invernal del NO₂ es la pauta contraria al NDVI. Ambas fuentes son complementarias y coherentes.

5. **NDVI_Gini:** captura desigualdad espacial que la media oculta. Ciudades con parques aislados muestran Gini alto a pesar de tener NDVI_Mean razonable.

6. **Panel ligeramente desbalanceado:** el **~X%** de los pares ciudad-año tienen los 12 meses cubiertos. Los efectos fijos ciudad en el modelo panel corrigen este sesgo.

### Variables seleccionadas para la regresión EKC

| Variable | Rol | Justificación EDA |
|----------|-----|------------------|
| `NDVI_Mean` (ln) | Variable dependiente | Alta cobertura, ciclo estacional bien definido |
| `gdp_pps_per_capita` (ln, ln²) | Variables explicativas EKC | Variable β₁, β₂ del modelo teórico |
| `Imperviousness_Mean` | Control de urbanización | Correlación negativa con NDVI_Slope |
| `NO2_Mean` | Variable dependiente alternativa | Estacionalidad inversa al NDVI |
| Efectos fijos ciudad | Control heterogeneidad | Panel desbalanceado requiere μᵢ |
| Efectos fijos año | Control shocks globales | COVID 2020, shocks energéticos |